# P1.7 — tune CRAG's thresholds

The defaults (±0.6) were guessed before the evaluator existed. The trained model puts
relevant passages at a mean of **+0.28** and irrelevant at **−0.54**, so ±0.6 would route
almost everything to "incorrect" and the CRAG row would lose to plain RAG for reasons
that have nothing to do with CRAG.

**Sidebar:** Accelerator **GPU T4 x2** · Internet **On** · Add Input: **verimem-data**,
plus the **notebook output** (or dataset) holding the trained evaluator.

## 1. Code and paths

In [ ]:
import os
import pathlib
import shutil
import subprocess
import sys

os.chdir("/kaggle/working")
REPO = pathlib.Path("/kaggle/working/VeriMem")
if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "-q", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "reset", "-q", "--hard", "origin/main"], check=True)
else:
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/TERABYTE05/VeriMem.git", str(REPO)], check=True
    )
os.chdir(REPO)
sys.path.insert(0, str(REPO))

DATA = pathlib.Path("/kaggle/working/data/averitec")
DATA.mkdir(parents=True, exist_ok=True)
os.environ["VERIMEM_DATA_DIR"] = "/kaggle/working/data"
os.environ["VERIMEM_CACHE_DIR"] = "/kaggle/working/cache"
os.environ["VERIMEM_RESULTS_DIR"] = "/kaggle/working/results"

for name in ("dev.json", "pool_dev.jsonl"):
    if not (DATA / name).exists():
        for found in pathlib.Path("/kaggle/input").rglob(name):
            shutil.copy(found, DATA / name)
            print("copied", name, f"{found.stat().st_size / 1e6:.1f} MB")
            break
    else:
        print("have", name)

for src_cache in pathlib.Path("/kaggle/input").rglob("cache/llm"):
    dst = pathlib.Path("/kaggle/working/cache/llm")
    if src_cache.is_dir() and not dst.exists():
        shutil.copytree(src_cache, dst)
        print(f"restored {len(list(dst.rglob('*.json')))} cached model calls")
        break

print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 2. Find the trained evaluator

Searches the attached inputs rather than hard-coding a path, since it may have come from
a dataset or from a notebook output.

In [ ]:
import pathlib

# Search working as well as input: a model trained in *this* session lives in
# /kaggle/working, and a notebook's own output is not mounted as its own input.
roots = [pathlib.Path("/kaggle/working"), pathlib.Path("/kaggle/input")]
candidates = [p.parent for root in roots if root.exists() for p in root.rglob("model/config.json")]
if not candidates:
    raise SystemExit(
        "No trained evaluator in /kaggle/working or /kaggle/input.\n"
        "Train one with evaluator.train, or attach the dataset or notebook output "
        "that holds results/<date>_t5-evaluator/model/.\n"
        "Without it the sweep falls back to a lexical stub, and thresholds tuned "
        "against word-counting do not transfer to the trained model."
    )
EVALUATOR = candidates[0]
print("evaluator:", EVALUATOR)
print(sorted(p.name for p in EVALUATOR.iterdir()))

## 3. Check it actually loads and separates

One relevant passage and one irrelevant one. The scores should straddle a gap; if they
do not, the model did not load properly and the sweep would tune noise.

In [ ]:
from evaluator.score import T5Scorer

scorer = T5Scorer(EVALUATOR)
claim = "The Kochi metro opened to the public in 2017."
scores = scorer.score(
    claim,
    [
        "The Kochi Metro began commercial operations in June 2017 after several delays.",
        "Penguins are flightless birds found mainly in the southern hemisphere.",
    ],
)
print(f"relevant   {scores[0]:+.3f}")
print(f"irrelevant {scores[1]:+.3f}")
assert scores[0] > scores[1], "the evaluator is not separating; check the model path"

## 4. The endpoint

The sweep makes model calls — one verify per branch, plus a rewrite. ~4.7 GB on a first
run; skip if a previous cell in this session already started ollama.

In [ ]:
import os
import shutil
import subprocess
import time
import urllib.request

if shutil.which("zstd") is None:
    subprocess.run(
        "apt-get -qq update && apt-get -qq install -y zstd", shell=True, capture_output=True
    )
if shutil.which("ollama") is None:
    subprocess.run(
        "curl -fsSL https://ollama.com/install.sh | sh", shell=True, capture_output=True, text=True
    )
OLLAMA = shutil.which("ollama")
assert OLLAMA, "ollama did not install - see the setup notebook"

try:
    urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
    print("already up")
except Exception:
    subprocess.Popen(
        [OLLAMA, "serve"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        start_new_session=True,
    )
    for _ in range(90):
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
            print("started")
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("ollama did not come up")

!ollama pull qwen2.5:7b

os.environ["VERIMEM_API_BASE_URL"] = "http://localhost:11434/v1"
os.environ["VERIMEM_API_KEY"] = "ollama"
os.environ["VERIMEM_API_MODEL"] = "qwen2.5:7b"

## 5. The sweep

150 claims x 3 branches = 450 pipeline runs. Each claim is run once per branch, after
which every grid point is a lookup rather than another run.

In [ ]:
!python -u -m evaluator.thresholds \
    --data /kaggle/working/data/averitec/dev.json \
    --store /kaggle/working/data/averitec/pool_dev.jsonl \
    --evaluator {EVALUATOR} \
    --limit 150

## 6. Save

`thresholds.json` is small. The cache is worth keeping too — those 450 runs are
expensive and every one is reusable.

In [ ]:
!cd /kaggle/working && zip -qr thresholds-out.zip results cache/llm cache/evaluator \
    -x "*/model/*" "*/checkpoints/*" && ls -lh thresholds-out.zip
from IPython.display import FileLink

FileLink("thresholds-out.zip")